# Pre-Requisites

## Creating Delta Table
- Create a catalog if thats not existing
- Create table
- Describe it

In [0]:
%sql

CREATE CATALOG IF NOT EXISTS merit_catalog;
USE CATALOG merit_catalog;
CREATE SCHEMA IF NOT EXISTS merit_catalog.quickstart_schema;


CREATE TABLE IF NOT EXISTS quickstart_schema.users_int(
   id INT,
   name STRING,
   dob DATE,
   email STRING,
   gender STRING,
   country STRING,
   region STRING,
   city STRING,
   asset INT,
   marital_status STRING
) USING DELTA;

DESCRIBE EXTENDED quickstart_schema.users_int;

# Transaction 01: Load data into Delta Table

In [0]:
df = spark.read.csv(
    path="/Volumes/merit_catalog/quickstart_schema/sandbox/dataset/user_dataset/users_001.csv",
    header=True,
    inferSchema=True,
)
df.limit(4).display()

In [0]:
df.write.mode("overwrite").saveAsTable("quickstart_schema.users_int")

## Read Delta Table

In [0]:
spark.read.table("quickstart_schema.users_int").display()

# Transaction 02: Append data into the Delta Table

In [0]:
from pyspark.sql.functions import col

df.filter(col("country") == "India").write.mode("append").saveAsTable("quickstart_schema.users_int")

spark.read.table("quickstart_schema.users_int").display()


# Transaction 03: Overwrite data into the Delta Table
- Data is overwritten and only the rows following the condition is accepted

In [0]:
from pyspark.sql.functions import col

df.filter(col("country") == "India").write.mode("overwrite").saveAsTable(
    "quickstart_schema.users_int"
)

spark.read.table("quickstart_schema.users_int").display()

# Transaction 04: Updating the Delta Table

In [0]:
%sql

UPDATE quickstart_schema.users_int
SET country = "IN"
WHERE country = "India"

## Read Delta Table after update

In [0]:
spark.read.table("quickstart_schema.users_int").display()

# Viewing Transaction log

In [0]:
from delta.tables import DeltaTable
table_name = "quickstart_schema.users_int"
delta_table = DeltaTable.forName(spark,table_name)
history_df = delta_table.history()
history_df.display()

# Read Specific Version
1. PySpark
2. SQL

## Using PySpark

In [0]:
spark.read.option("versionAsOf", 1).table("quickstart_schema.users_int").display()

## Using SQL

In [0]:
%sql
SELECT * from quickstart_schema.users_int VERSION AS OF 1;

# Read based on specific Timestamp

## Using SQL

In [0]:
%sql
SELECT * from quickstart_schema.users_int TIMESTAMP AS OF '2026-03-18T08:53:36.000+00:00';

# Restoring the old Delta Table
- Eg: Current version is 4, that will point to 1

In [0]:
%sql

RESTORE TABLE quickstart_schema.users_int VERSION AS OF 1;

SELECT * FROM quickstart_schema.users_int;

## Viewing Updated Transaction Log

In [0]:
from delta.tables import DeltaTable
table_name = "quickstart_schema.users_int"
delta_table = DeltaTable.forName(spark,table_name)
history_df = delta_table.history()
history_df.display()

## Reading updated Delta Table

In [0]:
spark.read.table("quickstart_schema.users_int").display()